In [1]:
import nltk
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import random
import string

nltk.download('punkt')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jbsan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jbsan\AppData\Roaming\nltk_data...


True

In [2]:
# Sample responses
responses = {
    'greet': ["Hello! How can I assist you today?", "Hi there! How can I help you?"],
    'appointment_inquiry': ["Sure, I can help you schedule an appointment. Please provide a preferred date and time."],
    'billing_payment_assistance': ["I can assist with billing inquiries. Please tell me your billing question."],
    'order_status_inquiry': ["I can check the status of your order. Please provide your order ID."],
    'product_information': ["We offer a variety of products and services. What specifically would you like to know?"],
    'fallback': ["Sorry, I did not understand your request. Please try again with a different query."]
}

# Example training data
training_data = {
    'greet': ["hello", "hi", "hey", "good morning", "good evening"],
    'appointment_inquiry': ["I want to schedule an appointment", "book a doctor's appointment", "what slots are available"],
    'billing_payment_assistance': ["help with my bill", "how to make a payment", "assist with billing"],
    'order_status_inquiry': ["what is the status of my order", "track my order", "where is my order"],
    'product_information': ["tell me about your services", "features of your products", "pricing for your services"]
}


In [3]:
def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    tokens = nltk.word_tokenize(text)
    return tokens

def get_synonyms(word):
    synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.append(lemma.name())
    return set(synonyms)


In [4]:
vectorizer = TfidfVectorizer(tokenizer=preprocess_text, stop_words='english')

def match_intent(user_input):
    intents = list(training_data.keys())
    corpus = [' '.join(training_data[intent]) for intent in intents]
    corpus.append(user_input)
    
    tfidf_matrix = vectorizer.fit_transform(corpus)
    cosine_sim = cosine_similarity(tfidf_matrix[-1], tfidf_matrix)
    
    best_match = np.argmax(cosine_sim[0][:-1])
    
    if cosine_sim[0][best_match] < 0.3:  # Threshold for similarity
        return 'fallback'
    
    return intents[best_match]


In [5]:
def chatbot_response(user_input):
    intent = match_intent(user_input)
    return random.choice(responses[intent])

# Chatbot interaction
print("Bot: Hello! How can I assist you today?")
while True:
    user_input = input("You: ").lower()
    if user_input == 'quit':
        print("Bot: Goodbye!")
        break
    else:
        print(f"Bot: {chatbot_response(user_input)}")

Bot: Hello! How can I assist you today?


c:\Users\jbsan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Bot: Sorry, I did not understand your request. Please try again with a different query.
Bot: Sorry, I did not understand your request. Please try again with a different query.
Bot: Sorry, I did not understand your request. Please try again with a different query.
